# CRISP-DM Phase 4: Modelling — Model 1, Track B
## Random Forest Baseline — All Features (No Feature Selection)
Pipeline: Cleaned data → Split → One-Hot Encode → StandardScaler → RF
No SelectKBest step. All encoded features are retained.
This track tests whether the full feature space outperforms the k=7 subset.
Baseline only — default parameters, no tuning.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, f1_score

df = joblib.load('../data/cleaned_data.pkl')

# Target column contains strings 'yes'/'no' — map to binary integers
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})

X = df.drop('personal_loan', axis=1)
y = df['personal_loan']

# Stratified split preserves 85/15 class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test  = X_test.copy()

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}")

Train: (4800, 11) | Test: (1200, 11)
Train class balance: {0: 0.85, 1: 0.15}


## One-Hot Encoding — Post-Split, Per Column
Categorical variables encoded after the split to prevent data leakage.
Each column is encoded individually and concatenated back, retaining
column headers as recommended in course notes.
Test set is aligned to training columns after encoding to handle any
categories present in one set but absent in the other.

In [2]:
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']

for col in categorical_cols:
    dummies_train = pd.get_dummies(X_train[col], prefix=col)
    X_train.drop([col], axis=1, inplace=True)
    X_train = pd.concat([X_train, dummies_train], axis=1)

    dummies_test = pd.get_dummies(X_test[col], prefix=col)
    X_test.drop([col], axis=1, inplace=True)
    X_test = pd.concat([X_test, dummies_test], axis=1)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print(f"Feature count after encoding: {X_train.shape[1]}")
print(f"Columns: {list(X_train.columns)}")

Feature count after encoding: 15
Columns: ['age', 'yrs_experience', 'family_size', 'income', 'mortgage_amt', 'credit_card_spend', 'share_trading_acct', 'fixed_deposit_acct', 'education_level_Advanced or Professional', 'education_level_Graduate', 'education_level_Undergraduate', 'credit_card_acct_no', 'credit_card_acct_yes', 'online_acct_no', 'online_acct_yes']


## Standardisation — Per Column, Fit on Training Data Only
Fitted exclusively on training data; transform applied to both sets.
Prevents test-set statistics from contaminating the training pipeline.

In [3]:
continuous_cols = ['age', 'yrs_experience', 'family_size',
                   'income', 'mortgage_amt', 'credit_card_spend']

scaler = StandardScaler()

for col in continuous_cols:
    scaler.fit(X_train[col].values.reshape(-1, 1))
    X_train[col] = scaler.transform(X_train[col].values.reshape(-1, 1))
    X_test[col]  = scaler.transform(X_test[col].values.reshape(-1, 1))

print(f"Scaled: {continuous_cols}")

Scaled: ['age', 'yrs_experience', 'family_size', 'income', 'mortgage_amt', 'credit_card_spend']


## Random Forest Baseline — All Features, Default Parameters
Contrast with notebook 03: full feature space vs k=7 selected features.

In [4]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
precision   = tp / (tp + fp)
recall      = tp / (tp + fn)
f1          = 2 * (precision * recall) / (precision + recall)
specificity = tn / (tn + fp)

print("=== RF Baseline — All Features ===")
print(f"\nConfusion Matrix:")
print(f"  TN: {tn}  FP: {fp}")
print(f"  FN: {fn}  TP: {tp}")
print(f"\nPrecision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-Score:    {f1:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"\nFull classification report:")
print(classification_report(y_test, y_pred))

=== RF Baseline — All Features ===

Confusion Matrix:
  TN: 972  FP: 48
  FN: 55  TP: 125

Precision:   0.7225
Recall:      0.6944
F1-Score:    0.7082
Specificity: 0.9529

Full classification report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      1020
           1       0.72      0.69      0.71       180

    accuracy                           0.91      1200
   macro avg       0.83      0.82      0.83      1200
weighted avg       0.91      0.91      0.91      1200

